<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Deep_Learning_Projects/CNN_Architecture/Tensorflow/CNN2_AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlexNet

<img src = "https://www.oreilly.com/api/v2/epubs/9781789956177/files/assets/ec08175c-5282-4be2-b6e7-6f2d99272166.png">

<img src = "https://i0.wp.com/syncedreview.com/wp-content/uploads/2017/05/13.png?resize=330%2C230&ssl=1">

### Import Libraries

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import Model, Sequential, layers, optimizers, losses, metrics

### Dataset Preparation

In [2]:
!pip install -q kaggle

In [3]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"hbahruz","key":"4f925bc35f1b4f8677fc6f8061cceb06"}'}

In [4]:
!kaggle datasets download -d akhiljethwa/forest-vs-desert

Dataset URL: https://www.kaggle.com/datasets/akhiljethwa/forest-vs-desert
License(s): CC-BY-NC-SA-4.0
 66% 5.00M/7.54M [00:00<00:00, 35.0MB/s]
100% 7.54M/7.54M [00:00<00:00, 42.0MB/s]


In [5]:
import zipfile

crab_species_zip = 'forest-vs-desert.zip'

def extract_zip(file_path, extract_to='.'):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

extract_zip(crab_species_zip, 'forest_vs_desert')

In [19]:
batch_size = 8
img_height = 227
img_width = 227

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,  # Use 20% of training data for validation
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'forest_vs_desert/Data',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=1024)
val_ds = val_ds.cache().prefetch(buffer_size=1024)

Found 802 files belonging to 2 classes.
Using 642 files for training.
Found 802 files belonging to 2 classes.
Using 160 files for validation.


### Model Building

In [20]:
class AlexNet(Model):
    def __init__(self, num_classes, in_channels = 3):
        super(AlexNet, self).__init__()

        self.seq = Sequential(
            [
                layers.Input(shape = (227, 227, in_channels)),
                layers.Conv2D(96, (11, 11), strides=(4, 4), activation='relu'),
                layers.MaxPooling2D((3, 3), strides=(2, 2)),
                layers.Conv2D(256, (5, 5), strides=(1, 1), padding='same', activation='relu'),
                layers.MaxPooling2D((3, 3), strides=(2, 2)),
                layers.Conv2D(384, (3, 3), strides=(1, 1), padding='same', activation='relu'),
                layers.Conv2D(384, (3, 3), strides=(1, 1), padding='same', activation='relu'),
                layers.Conv2D(256, (3, 3), strides=(1, 1), padding='same', activation='relu'),
                layers.MaxPooling2D((3, 3), strides=(2, 2)),
                layers.Flatten(),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(num_classes, activation='softmax')
            ]
        )

    def call(self, x):
        return self.seq(x)

### Using GPU, Training and Testing

In [21]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    strategy = tf.distribute.MirroredStrategy()
    print(f"Number of GPUs: {len(gpus)}")
else:
    strategy = tf.distribute.get_strategy()
    print("No GPUs available, using default strategy")

Number of GPUs: 1


In [22]:
with strategy.scope():
    model = AlexNet(num_classes=2)
    optimizer = optimizers.Adam(learning_rate=0.01)
    loss_fn = losses.SparseCategoricalCrossentropy()
    metric = metrics.SparseCategoricalAccuracy()

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=[metric])

In [23]:
model.fit(train_ds, epochs=5, validation_data=val_ds)

Epoch 1/5
81/81 [==============================] - 7s 49ms/step - loss: 35078240.0000 - sparse_categorical_accuracy: 0.4922 - val_loss: 0.6922 - val_sparse_categorical_accuracy: 0.5250
Epoch 2/5
81/81 [==============================] - 3s 35ms/step - loss: 8.0443 - sparse_categorical_accuracy: 0.5218 - val_loss: 0.6974 - val_sparse_categorical_accuracy: 0.5250
Epoch 3/5
81/81 [==============================] - 3s 33ms/step - loss: 0.7338 - sparse_categorical_accuracy: 0.4907 - val_loss: 0.6925 - val_sparse_categorical_accuracy: 0.5250
Epoch 4/5
81/81 [==============================] - 3s 33ms/step - loss: 0.7095 - sparse_categorical_accuracy: 0.5031 - val_loss: 0.7160 - val_sparse_categorical_accuracy: 0.5250
Epoch 5/5
81/81 [==============================] - 3s 34ms/step - loss: 0.7059 - sparse_categorical_accuracy: 0.4953 - val_loss: 0.7012 - val_sparse_categorical_accuracy: 0.4750
